In [10]:
import polars as pl
import opendp.prelude as dp

dp.enable_features("contrib")

TEST_SIZE = 10_000
    
# Polars dataframe with no rows so that the counts are all none.
test_data = {
    "A": [],
    "B": []
}
data_schema = pl.Schema({"A": pl.Int32, "B": pl.Int32})

# lots of keys where some portion should have negative counts with noise applied.
keys = pl.LazyFrame({"A": [i for i in range(TEST_SIZE)]})

# A very low budget for highly noisy results.
privacy_budget = 10e-10

context = dp.Context.compositor(
    data = pl.DataFrame(test_data, schema=data_schema),
    privacy_unit=dp.unit_of(changes=1),
    privacy_loss=dp.loss_of(epsilon=privacy_budget),
    split_evenly_over = 1,
)
result = (
    context.query()
    .groupby("A")
    .agg(dp.len())
    .with_keys(keys)
    .release()
    .collect())
result

TypeError: unrecognized carrier type: DataFrame